# CSET Assessment

In [ ]:
import os 
import json 
import regex as re
from collections import Counter
from datetime import datetime

from utils import read_word_list

In [3]:
papers = []
with open(os.path.join('..', 'data', 'arxiv_data.jsonl'), 'r') as file:
    for line in file: 
        papers.append(json.loads(line))

## Question 1: Identifying papers about agentic AI

### Question 1.A

In [ ]:
# load in pre-defined list of words
narrow_word_list = read_word_list(os.path.join('..', 'word_lists', 'agentic_ai_narrow.txt'))
broad_word_list = read_word_list(os.path.join('..', 'word_lists', 'agentic_ai_broad.txt'))

In [ ]:
narrow_matches = set()
broad_matches = set()

# compiling regular expressions outside of loop to improve runtime & remove redundant operations
narrow_regex_patterns = [re.compile(f'(?:\\b)?{word}(:?\\b)?') for word in narrow_word_list]
broad_regex_patterns = [re.compile(f'(?:\\b)?{word}(:?\\b)?') for word in broad_word_list]

for paper in papers:
    title = paper.get('title', '').lower()
    abstract = paper.get('abstract', '').lower()
    paper_id = paper.get('id')

    for pattern in narrow_regex_patterns:
        if pattern.search(title) or pattern.search(abstract):
            narrow_matches.add(paper_id)
            # only need to match one of the patterns to count 
            # cut down run time by ending loop if a match is already found
            break

    for pattern in broad_regex_patterns:
        if pattern.search(title) or pattern.search(abstract):
            broad_matches.add(paper_id)
            # only need to match one of the patterns to count 
            # cut down run time by ending loop if a match is already found
            break

### Question 1.B

In [40]:
cs_ma_papers = set([paper.get('id') for paper in papers if 'cs.ma' in paper.get('categories').lower()])

In [43]:
only_in_cs_ma = cs_ma_papers - broad_matches
only_in_broad_matches = broad_matches - cs_ma_papers
items_not_shared = broad_matches ^ cs_ma_papers

In [ ]:
# quick check that set math was done correctly
len(items_not_shared) == (len(only_in_broad_matches) + len(only_in_cs_ma))

True

What repositories were relevant papers in? 

In [65]:
repository_counts = Counter()

for paper in papers: 
    if paper.get('id') in broad_matches:
        repos = paper.get('categories_split')

        for repo in repos:
            repository_counts.update([repo])

In [67]:
repository_counts.most_common(10)

[('cs.AI', 4079),
 ('cs.MA', 1861),
 ('cs.CL', 1690),
 ('cs.LG', 1656),
 ('cs.SY', 893),
 ('eess.SY', 817),
 ('cs.RO', 727),
 ('cs.CR', 493),
 ('cs.SE', 481),
 ('math.OC', 478)]

## Question 2: Growth in research on agentic AI

In [ ]:
# get denominator (all publications in each year)
publications_by_year = Counter()
agentic_ai_publications_by_year = Counter()

for paper in papers:
    paper_id = paper.get('id')
    date = paper.get('created')
    year = datetime.fromisoformat(date).year

    publications_by_year.update([year])

    if paper_id in broad_matches:
        agentic_ai_publications_by_year.update([year])

In [61]:
pct_agentic_ai_publications = {}

for year in list(publications_by_year.keys()):
    agentic_ai_pubs = agentic_ai_publications_by_year.get(year, 0)
    overall_pubs = publications_by_year.get(year, 0)

    pct_agentic_ai_publications[year] = (agentic_ai_pubs / overall_pubs)

In [ ]:
pct_agentic_ai_publications

{2023: 0.00246421666954083,
 2022: 0.0017967417797859324,
 2024: 0.0051556258853256084,
 2025: 0.010197089828871042,
 2026: 0.016118374277517084,
 2021: 0.0019693590894610327,
 2019: 0.0015670325505459104,
 2020: 0.0017383034781659728,
 2017: 0.0008570816370259267,
 2011: 0.0005044772354647496,
 2018: 0.00145801376000486,
 2016: 0.0007461726489129037,
 2010: 0.0002291160702011639,
 2015: 0.0008426068512653631,
 2005: 0.00020873921514055107,
 2014: 0.0008339213548014626,
 2009: 0.00015057217426219636,
 2012: 0.0005381717536711002,
 2013: 0.0004441862848942495,
 2008: 0.0005466870763175158,
 1999: 0.00011803588290840416,
 2007: 0.00011604293588627792,
 2003: 0.0004195686833934715,
 1996: 0.0,
 1997: 0.0,
 2006: 0.0004586554842091469,
 2004: 0.00029781847963666143,
 2000: 0.0003202049311559398,
 1998: 0.0,
 1995: 0.0,
 1993: 0.0,
 2001: 0.0,
 2002: 0.00018223234624145787,
 1994: 0.0,
 1992: 0.0,
 1991: 0.0,
 1989: 0.0,
 1990: 0.0}